

| Secret name | Value |
|---|---|
| `DB_HOST` | e.g. `ep-xxxx.us-east-2.aws.neon.tech` (from Neon/Supabase) |
| `DB_PORT` | usually `5432` |
| `DB_NAME` | your database name |
| `DB_USER` | your database user |
| `DB_PASSWORD` | your database password |
| `JWT_SECRET` | any long random string (generate one in the next cell) |
| `SMTP_EMAIL` | your Gmail address |
| `SMTP_APP_PASSWORD` | 16-character Gmail **App Password** (not your real password) |
| `NGROK_AUTHTOKEN` | from https://dashboard.ngrok.com/get-started/your-authtoken |




In [1]:
!pip install -q streamlit psycopg2-binary PyJWT bcrypt python-dotenv email-validator pyngrok fastapi uvicorn python-multipart requests \
    langdetect ftfy emoji deep-translator vaderSentiment spacy pandas matplotlib transformers accelerate torch stopwordsiso deepface tf-keras opencv-python-headless mtcnn reportlab
!python -m spacy download xx_sent_ud_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 23.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 120.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 69.9 MB/s eta 0:00:00
   ━━━━━━

In [2]:
from google.colab import userdata

required_secrets = [
    "DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD",
    "JWT_SECRET", "SMTP_EMAIL", "SMTP_APP_PASSWORD", "NGROK_AUTHTOKEN",
]

values = {}
missing = []
for key in required_secrets:
    try:
        values[key] = userdata.get(key)
    except Exception:
        missing.append(key)

if missing:
    raise RuntimeError(
        f"Missing Colab secrets: {missing}. "
        f"Add them via the key icon in the left sidebar, then re-run this cell."
    )

env_content = f'''DB_HOST={values["DB_HOST"]}
DB_PORT={values["DB_PORT"]}
DB_NAME={values["DB_NAME"]}
DB_USER={values["DB_USER"]}
DB_PASSWORD={values["DB_PASSWORD"]}

JWT_SECRET={values["JWT_SECRET"]}
JWT_ALGORITHM=HS256
JWT_EXPIRY_MINUTES=60

SMTP_HOST=smtp.gmail.com
SMTP_PORT=587
SMTP_EMAIL={values["SMTP_EMAIL"]}
SMTP_APP_PASSWORD={values["SMTP_APP_PASSWORD"]}

OTP_EXPIRY_MINUTES=10
'''

with open(".env", "w") as f:
    f.write(env_content)

print("Wrote .env with", len(values), "secrets loaded.")

Wrote .env with 9 secrets loaded.


In [3]:
%%writefile db.py
import os, psycopg2
from psycopg2.extras import RealDictCursor
from contextlib import contextmanager
from dotenv import load_dotenv
load_dotenv()

CFG = dict(host=os.getenv("DB_HOST"), port=os.getenv("DB_PORT", "5432"),
           dbname=os.getenv("DB_NAME"), user=os.getenv("DB_USER"),
           password=os.getenv("DB_PASSWORD"), sslmode="require")

@contextmanager
def cursor(commit=False):
    conn = psycopg2.connect(**CFG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    try:
        yield cur
        if commit: conn.commit()
    finally:
        cur.close(); conn.close()

def init_db():
    with cursor(commit=True) as cur:
        cur.execute("""CREATE TABLE IF NOT EXISTS users (
            id SERIAL PRIMARY KEY, username VARCHAR(50) UNIQUE, email VARCHAR(255) UNIQUE,
            password_hash VARCHAR(255), is_verified BOOLEAN DEFAULT FALSE,
            role VARCHAR(20) NOT NULL DEFAULT 'employee')""")
        cur.execute("""ALTER TABLE users ADD COLUMN IF NOT EXISTS role VARCHAR(20) NOT NULL DEFAULT 'employee'""")
        cur.execute("""CREATE TABLE IF NOT EXISTS otp_codes (
            id SERIAL PRIMARY KEY, email VARCHAR(255), code VARCHAR(6),
            purpose VARCHAR(20), expires_at TIMESTAMP, used BOOLEAN DEFAULT FALSE)""")

        cur.execute("""CREATE TABLE IF NOT EXISTS mood_logs (
            id SERIAL PRIMARY KEY,
            user_id INTEGER NOT NULL REFERENCES users(id) ON DELETE CASCADE,
            mood_date DATE NOT NULL DEFAULT CURRENT_DATE,
            sentiment VARCHAR(20),
            emotion VARCHAR(30),
            compound_score REAL,
            confidence REAL,
            journal_text TEXT,
            source VARCHAR(10) NOT NULL DEFAULT 'manual',
            created_at TIMESTAMP NOT NULL DEFAULT NOW())""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS sentiment VARCHAR(20)""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS emotion VARCHAR(30)""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS compound_score REAL""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS journal_text TEXT""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS source VARCHAR(10) NOT NULL DEFAULT 'manual'""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS confidence REAL""")
        cur.execute("""CREATE INDEX IF NOT EXISTS idx_mood_logs_user_date
            ON mood_logs(user_id, mood_date)""")


MOOD_LABELS = ["Happy", "Neutral", "Sad", "Stress", "Angry", "Fear"]

MOOD_EMOJI = {
    "Happy": "\U0001F60A",
    "Neutral": "\U0001F610",
    "Sad": "\U0001F622",
    "Stress": "\U0001F62B",
    "Angry": "\U0001F620",
    "Fear": "\U0001F628",
}


def save_manual_mood(user_id, mood_label):
    with cursor(commit=True) as cur:
        cur.execute(
            """INSERT INTO mood_logs (user_id, sentiment, source)
               VALUES (%s, %s, 'manual')""",
            (user_id, mood_label),
        )

def save_mood_log(user_id, sentiment, emotion, compound_score, journal_text, confidence=None):
    mood_label = emotion if emotion in MOOD_LABELS else "Neutral"
    with cursor(commit=True) as cur:
        cur.execute(
            """INSERT INTO mood_logs (user_id, sentiment, emotion, compound_score, confidence, journal_text, source)
               VALUES (%s, %s, %s, %s, %s, %s, 'nlp')""",
            (user_id, mood_label, emotion, compound_score, confidence, journal_text),
        )

def get_mood_logs_for_month(user_id, year, month):
    with cursor() as cur:
        cur.execute(
            """SELECT DISTINCT ON (mood_date) mood_date, sentiment, emotion, compound_score, confidence, created_at
               FROM mood_logs
               WHERE user_id = %s
                 AND EXTRACT(YEAR FROM mood_date) = %s
                 AND EXTRACT(MONTH FROM mood_date) = %s
               ORDER BY mood_date, created_at DESC""",
            (user_id, year, month),
        )
        return cur.fetchall()

def get_user_mood_history(user_id, limit=200):
    with cursor() as cur:
        cur.execute(
            """SELECT mood_date, sentiment, emotion, compound_score, confidence, journal_text, source, created_at
               FROM mood_logs
               WHERE user_id = %s
               ORDER BY created_at DESC
               LIMIT %s""",
            (user_id, limit),
        )
        return cur.fetchall()

def get_all_employee_mood_logs(limit_days=30):
    with cursor() as cur:
        cur.execute(
            """SELECT u.username, u.email, m.mood_date, m.sentiment, m.emotion, m.compound_score, m.confidence, m.created_at
               FROM mood_logs m
               JOIN users u ON u.id = m.user_id
               WHERE u.role = 'employee'
                 AND m.mood_date >= CURRENT_DATE - (%s || ' days')::interval
               ORDER BY m.mood_date DESC, u.username""",
            (limit_days,),
        )
        return cur.fetchall()

def get_latest_mood_per_employee():
    with cursor() as cur:
        cur.execute(
            """SELECT DISTINCT ON (u.id) u.username, u.email, m.mood_date, m.sentiment, m.emotion, m.confidence, m.created_at
               FROM users u
               JOIN mood_logs m ON m.user_id = u.id
               WHERE u.role = 'employee'
               ORDER BY u.id, m.created_at DESC"""
        )
        return cur.fetchall()


def save_face_scan(user_id, emotion, confidence):
    """Saves a face scan mood log"""
    mood_label = "Normal"
    if emotion in ["happy", "joy", "amazing"]: mood_label = "Happy"
    elif emotion in ["sad", "sadness"]: mood_label = "Sad"
    elif emotion in ["angry", "anger"]: mood_label = "Angry"

    with cursor(commit=True) as cur:
        cur.execute(
            """INSERT INTO mood_logs (user_id, sentiment, emotion, compound_score, confidence, journal_text, source)
               VALUES (%s, %s, %s, %s, %s, %s, 'face')""",
            (user_id, mood_label, emotion.capitalize(), 0.0, float(confidence), 'Face Scan', )
        )


Writing db.py


In [4]:
from db import cursor

with cursor(commit=True) as cur:
    cur.execute("UPDATE mood_logs SET sentiment = 'Happy' WHERE sentiment = 'Amazing'")
    cur.execute("UPDATE mood_logs SET sentiment = 'Neutral' WHERE sentiment = 'Normal'")
print("Remapped legacy Amazing/Normal rows to Happy/Neutral.")

Remapped legacy Amazing/Normal rows to Happy/Neutral.


In [5]:
%%writefile recommendations.py
"""
recommendations.py
Lightweight, dependency-free (no torch/spacy) home for the wellness
recommendation engine, shared by:
  - nlp_pipeline.py  -> get_recommendation() for a single journal entry
  - app.py           -> get_period_recommendation() for a Dashboard
                         date-range PDF export summary

Kept separate from nlp_pipeline.py so app.py (a plain Streamlit process)
doesn't need to import the heavy NLP stack just to build a report.
"""

# ---------------------------------------------------------------------------
# Wellness recommendation engine
#
# Simple rule-based recommender: maps a detected emotion label to a small set
# of curated wellness suggestions. This mirrors what a real MoodMentor-style
# system would do -- detected emotional state -> mapped intervention -- just
# without a database-backed content repository behind it.
#
# The confidence score (0-1) is used to pick *how* urgent/serious the
# suggestion should be for Sad/Stress/Angry/Fear:
#   - low confidence  (< 0.4): the model isn't very sure, so keep it light/generic
#   - medium confidence (0.4-0.7): a normal, matched coping suggestion
#   - high confidence (>= 0.7): the emotion signal is strong, so nudge more
#     firmly towards professional/structured support
#
# Happy and Neutral don't need an urgency ladder -- they always get an
# encouraging or maintenance-style tip instead.
# ---------------------------------------------------------------------------

WELLNESS_RECOMMENDATIONS = {
    "Happy": [
        "Great to see you're feeling good! Take a moment to note what contributed to this — it helps to recognize your own positive patterns.",
        "Keep this momentum going: consider sharing your positive energy with a colleague or teammate today.",
    ],
    "Neutral": [
        "A calm, steady mood is a good baseline. A short 5-minute walk or stretch break can help maintain it.",
        "Nothing urgent here — this could be a good time to plan your day or check in on a personal goal.",
    ],
    "Sad": {
        "low": "It looks like there might be a touch of sadness here. Consider writing a bit more in your journal about what's on your mind.",
        "medium": "Try a short guided breathing exercise (4 seconds in, 4 seconds hold, 4 seconds out) or step outside for a few minutes.",
        "high": "This seems like a strong low mood. Please consider talking to a trusted colleague, friend, or your HR/EAP wellness contact today.",
    },
    "Stress": {
        "low": "A little stress is normal — try a quick 2-minute breathing break before your next task.",
        "medium": "Consider breaking your current task into smaller steps, and take a 10-minute break away from your screen.",
        "high": "Your stress signal looks high. Try a longer break, deep breathing, or a short walk, and consider flagging your workload to your manager or HR.",
    },
    "Angry": {
        "low": "A bit of frustration is showing. A short pause before responding to anything stressful can help.",
        "medium": "Try stepping away for 5-10 minutes before continuing. Cognitive reframing — writing down the situation objectively — can help too.",
        "high": "This reads as strong frustration or anger. Please take a proper break away from the trigger, and consider talking it through with someone you trust or your HR/EAP contact.",
    },
    "Fear": {
        "low": "A little anxiety is showing. Grounding techniques (naming 5 things you can see, 4 you can hear) can help settle it.",
        "medium": "Try a short guided breathing or grounding exercise, and write down specifically what's worrying you — it often feels more manageable on paper.",
        "high": "This looks like a strong fear/anxiety signal. Please consider reaching out to a trusted colleague, your HR/EAP program, or a mental health professional.",
    },
}

# Maps the 5-point manual mood-picker label (db.MOOD_LABELS) onto the same
# 6-label emotion vocabulary above, so entries with no NLP/emotion data
# (manual mood taps) can still be folded into a recommendation.
MOOD_TO_EMOTION_BUCKET = {
    "Amazing": "Happy",
    "Happy": "Happy",
    "Normal": "Neutral",
    "Sad": "Sad",
    "Angry": "Angry",
}


def _confidence_bucket(confidence: float) -> str:
    """Buckets a 0-1 confidence score into low / medium / high urgency."""
    if confidence is None:
        return "medium"
    if confidence < 0.4:
        return "low"
    if confidence < 0.7:
        return "medium"
    return "high"


def get_recommendation(
    emotion_label: str,
    confidence: float = None,
    sentiment: str = None,
    sentiment_score: float = None,
) -> str:
    """
    Returns a wellness suggestion string, combining both classifiers:

    - `emotion_label` / `confidence` come from the BERT emotion model
      (Happy, Sad, Stress, Angry, Fear, Neutral).
    - `sentiment` / `sentiment_score` come from VADER (Positive, Negative,
      Neutral + a compound score from -1 to 1).

    These two models are trained independently and can disagree -- e.g. BERT
    says "Neutral" while VADER's compound score is clearly negative. Relying
    on the emotion label alone would then give a generic "maintenance" tip
    for text that actually reads negative.

    Fix: if BERT's top emotion is "Neutral" but VADER disagrees and calls the
    text "Negative", we treat it as mild Sad/Stress instead of Neutral, using
    the *sentiment* score for urgency instead of the (less reliable, in this
    case) emotion confidence. Otherwise, emotion label + emotion confidence
    drive the recommendation as before.
    """
    effective_label = emotion_label
    effective_confidence = confidence

    if emotion_label == "Neutral" and sentiment == "Negative":
        effective_label = "Sad"
        magnitude = abs(sentiment_score) if sentiment_score is not None else 0.3
        effective_confidence = magnitude  # -1..1 magnitude reused as 0..1 bucket input

    entry = WELLNESS_RECOMMENDATIONS.get(effective_label)
    if entry is None:
        return "Take a moment to check in with yourself today."

    if isinstance(entry, list):
        # Happy / Neutral (and genuinely neutral-sentiment text): no urgency
        # ladder, just rotate suggestions.
        import random
        return random.choice(entry)

    bucket = _confidence_bucket(effective_confidence)
    return entry[bucket]


def get_period_recommendation(entries: list[dict]) -> str:
    """
    Builds a short 2-3 sentence wellness summary for a *set* of mood_logs
    rows (e.g. everything within a Dashboard date-range export), rather
    than a single journal entry.

    Each `entries` item is expected to look like a row from
    db.get_user_mood_history(): at minimum `sentiment` (the 5-point mood
    label), and optionally `emotion` + `confidence` (present only for
    source == 'nlp' journal entries).

    Prefers the richer emotion/confidence data where available and falls
    back to the manual mood-picker label (mapped onto the same bucket
    vocabulary) otherwise, so a period made up of only emoji taps still
    gets a sensible recommendation.
    """
    if not entries:
        return "No entries were logged in this period yet."

    bucket_counts: dict[str, int] = {}
    bucket_confidences: dict[str, list[float]] = {}

    for e in entries:
        if e.get("source") == "nlp" and e.get("emotion"):
            bucket = e["emotion"]
            conf = e.get("confidence")
        else:
            bucket = MOOD_TO_EMOTION_BUCKET.get(e.get("sentiment"), "Neutral")
            conf = None

        bucket_counts[bucket] = bucket_counts.get(bucket, 0) + 1
        if conf is not None:
            bucket_confidences.setdefault(bucket, []).append(conf)

    total = sum(bucket_counts.values())
    dominant_bucket = max(bucket_counts, key=bucket_counts.get)
    dominant_count = bucket_counts[dominant_bucket]
    pct = round(100 * dominant_count / total)

    confs = bucket_confidences.get(dominant_bucket)
    avg_conf = sum(confs) / len(confs) if confs else None

    tip = get_recommendation(dominant_bucket, avg_conf)

    overview = (
        f"Over this period, {dominant_bucket.lower()} was your most common state "
        f"({dominant_count} of {total} entries, {pct}%)."
    )
    closing = "Keep logging regularly so trends like this are easier to catch early."

    return f"{overview} {tip} {closing}"


Writing recommendations.py


In [6]:
%%writefile auth.py
import os, jwt, bcrypt, random, string
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv
from db import cursor
load_dotenv()

SECRET = os.getenv("JWT_SECRET")

def hash_pw(pw): return bcrypt.hashpw(pw.encode(), bcrypt.gensalt()).decode()
def check_pw(pw, h): return bcrypt.checkpw(pw.encode(), h.encode())

def make_token(user):
    payload = {"id": user["id"], "username": user["username"], "email": user["email"],
               "role": user.get("role", "employee"),
               "exp": datetime.now(timezone.utc) + timedelta(hours=1)}
    return jwt.encode(payload, SECRET, algorithm="HS256")

def read_token(token):
    try: return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError: return None

def get_user(email):
    with cursor() as cur:
        cur.execute("SELECT * FROM users WHERE email=%s", (email,))
        return cur.fetchone()

def username_taken(username):
    with cursor() as cur:
        cur.execute("SELECT 1 FROM users WHERE username=%s", (username,))
        return cur.fetchone() is not None

def create_user(username, email, pw, role="employee"):
    with cursor(commit=True) as cur:
        cur.execute("INSERT INTO users (username,email,password_hash,role) VALUES (%s,%s,%s,%s)",
                    (username, email, hash_pw(pw), role))

def verify_user(email):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET is_verified=TRUE WHERE email=%s", (email,))

def set_password(email, pw):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET password_hash=%s WHERE email=%s", (hash_pw(pw), email))

def new_otp():
    return "".join(random.choices(string.digits, k=6))

def save_otp(email, code, purpose):
    exp = datetime.now(timezone.utc) + timedelta(minutes=10)
    with cursor(commit=True) as cur:
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE email=%s AND purpose=%s", (email, purpose))
        cur.execute("INSERT INTO otp_codes (email,code,purpose,expires_at) VALUES (%s,%s,%s,%s)",
                    (email, code, purpose, exp))

def check_otp(email, code, purpose):
    with cursor(commit=True) as cur:
        cur.execute("""SELECT * FROM otp_codes WHERE email=%s AND purpose=%s AND used=FALSE
                       ORDER BY id DESC LIMIT 1""", (email, purpose))
        row = cur.fetchone()
        if not row or row["code"] != code:
            return False
        now = datetime.now(row["expires_at"].tzinfo) if row["expires_at"].tzinfo else datetime.now()
        if now > row["expires_at"]:
            return False
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE id=%s", (row["id"],))
        return True

Writing auth.py


In [7]:
%%writefile email_utils.py
import os, smtplib
from email.mime.text import MIMEText
from dotenv import load_dotenv
load_dotenv()

HOST, PORT = "smtp.gmail.com", 587
EMAIL = os.getenv("SMTP_EMAIL")
APP_PW = os.getenv("SMTP_APP_PASSWORD")

def send_otp(to_email, code, purpose):
    subject = "Your Verification Code" if purpose == "signup" else "Your Password Reset Code"
    msg = MIMEText(f"Your code is: {code}\nExpires in 10 minutes.")
    msg["From"], msg["To"], msg["Subject"] = EMAIL, to_email, subject
    try:
        with smtplib.SMTP(HOST, PORT, timeout=15) as s:
            s.starttls()
            s.login(EMAIL, APP_PW)
            s.sendmail(EMAIL, to_email, msg.as_string())
        return True, "sent"
    except Exception as e:
        return False, str(e)

Writing email_utils.py


In [8]:
%%writefile app.py
import os, re, io, calendar
from datetime import date, datetime
import requests, streamlit as st
import matplotlib.pyplot as plt
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from db import (init_db, save_mood_log, save_manual_mood, MOOD_LABELS, MOOD_EMOJI,
                 get_mood_logs_for_month, get_user_mood_history,
                 get_all_employee_mood_logs, get_latest_mood_per_employee, save_face_scan)
from recommendations import get_period_recommendation
from auth import (make_token, read_token, get_user, username_taken, create_user,
                   verify_user, set_password, check_pw, new_otp, save_otp, check_otp)
from email_utils import send_otp

st.set_page_config(page_title="MoodMentor", layout="wide")

BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8000")

BRAND_GREEN = "#1DBF73"
BRAND_GREEN_DARK = "#159c5e"
INK = "#1f2937"
MUTED = "#6b7280"
BG = "#f5f7f6"

MOOD_STYLE = {
    "Happy":   {"emoji": MOOD_EMOJI["Happy"],   "color": "#2ecc71"},
    "Neutral": {"emoji": MOOD_EMOJI["Neutral"], "color": "#3498db"},
    "Sad":     {"emoji": MOOD_EMOJI["Sad"],     "color": "#e67e22"},
    "Stress":  {"emoji": MOOD_EMOJI["Stress"],  "color": "#f1c40f"},
    "Angry":   {"emoji": MOOD_EMOJI["Angry"],   "color": "#e74c3c"},
    "Fear":    {"emoji": MOOD_EMOJI["Fear"],    "color": "#9b59b6"},
}
def style_for(label):
    return MOOD_STYLE.get(label, {"emoji": "", "color": "#bdbdbd"})

MOOD_TO_NUM = {"Happy": 2, "Neutral": 0, "Sad": -1, "Stress": -1, "Angry": -2, "Fear": -2}

def inject_css():
    st.markdown("""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800;900&display=swap');

        header {visibility: hidden;}

        /* 1. LOGIN / FIRST PAGE BACKGROUND */
        .stApp {
            background: url('...') center center no-repeat !important;
            background-size: cover !important;
            background-attachment: fixed !important;
            font-family: 'Plus Jakarta Sans', sans-serif !important;
        }

        /* 2. HOME PAGE BACKGROUND */
        .stApp:has([data-testid="stSidebar"]) {
            background: url('...') center center repeat !important;
            background-size: cover !important;
            background-attachment: fixed !important;
            color: #1e293b !important;
        }

        .block-container {
            background: transparent !important;
            box-shadow: none !important;
            border: none !important;
            backdrop-filter: none !important;
            -webkit-backdrop-filter: none !important;
            padding: 3rem !important;
            max-width: 1100px !important;
        }

        /* 3. THE SIDEBAR (Yoga Couple with Soft Blend) */
        [data-testid="stSidebar"] {
            background: linear-gradient(to bottom, rgba(245,240,230,0.6) 0%, rgba(245,240,230,0.9) 100%), url('...') center top no-repeat !important;
            background-size: cover !important;
            border-right: 1px solid rgba(0,0,0,0.05) !important;
            box-shadow: 5px 0 15px rgba(0,0,0,0.03) !important;
        }

        /* Sidebar Logo Treatment for "Mood Mentor" */
        [data-testid="stSidebar"] .stMarkdown h1, [data-testid="stSidebar"] .stMarkdown h2, [data-testid="stSidebar"] .stMarkdown h3 {
            font-weight: 900 !important;
            font-size: 26px !important;
            letter-spacing: -1px !important;
            background: linear-gradient(90deg, #b45309, #d97706) !important;
            -webkit-background-clip: text !important;
            -webkit-text-fill-color: transparent !important;
            text-transform: uppercase !important;
            margin-bottom: 30px !important;
            padding-bottom: 10px !important;
            border-bottom: 2px solid rgba(180, 83, 9, 0.2) !important;
            text-shadow: none !important;
        }

        [data-testid="stSidebar"] span, [data-testid="stSidebar"] p, [data-testid="stSidebar"] label, [data-testid="stSidebar"] div {
            color: #1e293b !important;
            font-weight: 800 !important;
            font-size: 14px !important;
            text-shadow: none !important;
        }

        [data-testid="stSidebar"] div[role="radiogroup"] > label {
            background: rgba(255,255,255,0.4) !important;
            backdrop-filter: blur(5px) !important;
            padding: 10px 16px !important;
            border-radius: 12px !important;
            margin-bottom: 6px !important;
            border: 1px solid rgba(255,255,255,0.6) !important;
            transition: all 0.2s ease !important;
        }
        [data-testid="stSidebar"] div[role="radiogroup"] > label:hover {
            background: rgba(255,255,255,0.8) !important;
        }
        [data-testid="stSidebar"] div[role="radiogroup"] > label[data-checked="true"] {
            background: rgba(255,255,255,0.95) !important;
            border-left: 4px solid #ef4444 !important;
            box-shadow: 0 4px 10px rgba(0,0,0,0.05) !important;
        }
        [data-testid="stSidebar"] div[role="radiogroup"] > label[data-checked="true"] span,
        [data-testid="stSidebar"] div[role="radiogroup"] > label[data-checked="true"] div {
            color: #0f172a !important;
        }

        [data-testid="stSidebar"] div[data-testid="stCaptionContainer"] {
            background: rgba(255,255,255,0.4) !important;
            backdrop-filter: blur(5px) !important;
            padding: 10px 15px !important;
            border-radius: 12px !important;
            margin-top: 15px !important;
            border: 1px solid rgba(255,255,255,0.5) !important;
        }

        /* 4. Typography Main Area */
        h1, h2, h3, h4, h5, h6 {
            color: #0f172a !important;
            font-weight: 800 !important;
            letter-spacing: -1px !important;
            text-shadow: none !important;
        }
        p, li, .stMarkdown p {
            color: #1e293b !important;
            font-weight: 600 !important;
            text-shadow: none !important;
        }
        h1 span, h2 span, h3 span {
            color: #0f172a !important;
            text-shadow: none !important;
        }

        /* 5. Metric Cards & Inner Elements - 3D Marble Stone Blocks */
        .mm-metric {
            background: url('...') center center no-repeat !important;
            background-size: cover !important;
            border-radius: 12px !important;
            padding: 24px !important;
            border: 1px solid rgba(255,255,255,0.5) !important;
            box-shadow: 6px 8px 15px rgba(0, 0, 0, 0.15), -4px -4px 10px rgba(255, 255, 255, 0.7), inset 2px 2px 5px rgba(255, 255, 255, 0.5) !important;
            transition: all 0.2s ease !important;
        }
        .mm-metric:hover {
            transform: translateY(-3px);
            box-shadow: 8px 10px 20px rgba(0, 0, 0, 0.2), -4px -4px 10px rgba(255, 255, 255, 0.8), inset 2px 2px 5px rgba(255, 255, 255, 0.5) !important;
        }
        .mm-metric .mm-value { color: #0f172a !important; font-size: 32px !important; font-weight: 800 !important; line-height: 1.2 !important; margin-top: 5px !important; letter-spacing: -0.5px; }
        .mm-metric .mm-label { color: #b45309 !important; font-size: 12px !important; font-weight: 800 !important; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 0px; }
        .mm-metric p { color: #1e293b !important; }

        /* 6. Radio Buttons (Emoji Wood Shelf) */
        .stApp > header + div > div > div > div > div > div > div > div > div > div > div > div > div.stRadio > div[role="radiogroup"] {
            background: url('...') center center no-repeat !important;
            background-size: cover !important;
            border-radius: 16px !important;
            padding: 20px 20px !important;
            box-shadow: 0 15px 35px rgba(0,0,0,0.3), inset 0px 5px 15px rgba(255,255,255,0.1) !important;
            border-bottom: 8px solid #3e2723 !important;
            border-top: 1px solid rgba(255,255,255,0.2) !important;
            display: flex !important;
            gap: 10px !important;
            flex-wrap: wrap !important;
            justify-content: space-around !important;
            margin-top: 20px !important;
            margin-bottom: 20px !important;
        }

        .stApp > header + div > div > div > div > div > div > div > div > div > div > div > div > div.stRadio > div[role="radiogroup"] > label {
            background: transparent !important;
            border-radius: 0px !important;
            padding: 10px !important;
            border: none !important;
            box-shadow: none !important;
            margin-right: 0px !important;
            transition: all 0.2s ease !important;
            text-align: center !important;
            flex: 1 1 0px !important;
            min-width: 80px !important;
        }
        .stApp > header + div > div > div > div > div > div > div > div > div > div > div > div > div.stRadio > div[role="radiogroup"] > label p {
            color: #d1d5db !important;
            font-weight: 700 !important;
            text-shadow: 0px 1px 2px rgba(0,0,0,0.8) !important;
            font-size: 14px !important;
            margin-top: 10px !important;
        }
        .stApp > header + div > div > div > div > div > div > div > div > div > div > div > div > div.stRadio > div[role="radiogroup"] > label:hover {
            transform: translateY(-5px);
        }
        .stApp > header + div > div > div > div > div > div > div > div > div > div > div > div > div.stRadio > div[role="radiogroup"] > label[data-checked="true"] {
            background: rgba(255,255,255,0.1) !important;
            border-radius: 12px !important;
            box-shadow: 0 0 20px rgba(255,255,255,0.2) !important;
        }
        .stApp > header + div > div > div > div > div > div > div > div > div > div > div > div > div.stRadio > div[role="radiogroup"] > label[data-checked="true"] p {
            color: #ffffff !important;
        }

        /* 7. Glossy Purple "Save mood" Button */
        .stFormSubmitButton>button, div[data-testid="stVerticalBlock"] > div > div > div > div.stButton > button {
            background: radial-gradient(circle at top, #a855f7, #6b21a8) !important;
            border-radius: 30px !important;
            padding: 0.6rem 2.5rem !important;
            border: 2px solid rgba(255,255,255,0.4) !important;
            box-shadow: 0 10px 25px rgba(107, 33, 168, 0.4), inset 0px 4px 10px rgba(255,255,255,0.3) !important;
            transition: all 0.2s ease !important;
            color: #ffffff !important;
            font-weight: 800 !important;
            text-shadow: 0px 1px 3px rgba(0,0,0,0.5) !important;
        }
        .stFormSubmitButton>button:hover, div[data-testid="stVerticalBlock"] > div > div > div > div.stButton > button:hover {
            transform: translateY(-2px);
            box-shadow: 0 15px 35px rgba(107, 33, 168, 0.6), inset 0px 4px 10px rgba(255,255,255,0.5) !important;
            background: radial-gradient(circle at top, #c084fc, #7e22ce) !important;
        }

        /* CRITICAL FIX: Sign Up / Forgot Password Buttons on Login Page AND Sidebar Logout */
        button[kind="secondary"] {
            background: rgba(255,255,255,0.95) !important;
            border: 1px solid rgba(0,0,0,0.1) !important;
            box-shadow: 0 4px 10px rgba(0,0,0,0.15) !important;
            border-radius: 12px !important;
            transition: all 0.2s ease !important;
        }
        button[kind="secondary"] * {
            color: #0f172a !important;
            font-weight: 800 !important;
            text-shadow: none !important;
        }
        button[kind="secondary"]:hover {
            background: #ffffff !important;
            transform: translateY(-2px);
            box-shadow: 0 8px 15px rgba(0,0,0,0.2) !important;
        }

        /* 8. Calendar Cells Fix - Marble block aesthetic */
        div[title] {
            background: url('...') center center no-repeat !important;
            background-size: cover !important;
            border-radius: 12px !important;
            padding: 8px !important;
            box-shadow: 4px 4px 10px rgba(0, 0, 0, 0.1), inset 1px 1px 3px rgba(255, 255, 255, 0.6) !important;
            border: 1px solid rgba(255,255,255,0.5) !important;
        }
        div[title] div {
            color: #0f172a !important;
            font-weight: 800 !important;
        }

        /* Journal Expanders */
        [data-testid="stExpander"] {
            background: rgba(255, 255, 255, 0.7) !important;
            border-radius: 12px !important;
            border: 1px solid rgba(0,0,0,0.05) !important;
            box-shadow: 0 4px 10px rgba(0,0,0,0.05) !important;
        }
        [data-testid="stExpander"] p, [data-testid="stExpander"] span {
            color: #0f172a !important;
        }

        /* 9. Login Page "Shock" Factor */
        .welcome-box, .auth-card {
            background: transparent !important;
            padding: 0 !important;
            border: none !important;
            box-shadow: none !important;
            margin: 0 !important;
        }

        [data-testid="stForm"] {
            position: relative;
            background: #ffffff !important;
            border-radius: 20px !important;
            border: 1px solid #e2e8f0 !important;
            box-shadow: 0 10px 25px -5px rgba(0, 0, 0, 0.1), 0 8px 10px -6px rgba(0, 0, 0, 0.1) !important;
            overflow: hidden !important;
            padding: 24px !important;
            padding-top: 220px !important;
            margin-top: 1rem !important;
        }

        [data-testid="stForm"]::before {
            content: "";
            position: absolute;
            top: 0;
            left: 0;
            width: 100%;
            height: 200px;
            background: url('...') center center no-repeat;
            background-size: cover;
            border-bottom: 4px solid #f59e0b;
            z-index: 0;
        }

        [data-testid="stForm"] > * {
            position: relative;
            z-index: 1;
        }

        [data-testid="stForm"] p, [data-testid="stForm"] h1, [data-testid="stForm"] h2, [data-testid="stForm"] h3 {
            color: #0f172a !important;
            text-shadow: none !important;
        }

    </style>
    """, unsafe_allow_html=True)

def donut_chart(counts: dict, size=2.6):
    labels, values, colors = [], [], []
    for k, v in counts.items():
        if v > 0:
            labels.append(k); values.append(v)
            colors.append(style_for(k)["color"])
    if not values:
        return None
    fig, ax = plt.subplots(figsize=(size, size))
    ax.pie(values, colors=colors, startangle=90, wedgeprops=dict(width=0.38, edgecolor="white"))
    ax.set(aspect="equal")
    fig.patch.set_alpha(0.0)
    return fig

def metric_tile(label, value, sub=None):
    sub_html = f"<div class='mm-sub'>{sub}</div>" if sub else ""
    st.markdown(
        f"<div class='mm-metric'><div class='mm-label'>{label}</div>"
        f"<div class='mm-value'>{value}</div>{sub_html}</div>",
        unsafe_allow_html=True,
    )

def build_pdf_report(username, start_d, end_d, entries, recommendation_text):
    buf = io.BytesIO()
    doc = SimpleDocTemplate(buf, pagesize=letter, topMargin=48, bottomMargin=48)
    styles = getSampleStyleSheet()
    story = []

    story.append(Paragraph("MoodMentor Wellness Report", styles["Title"]))
    story.append(Paragraph(f"{username} &nbsp;|&nbsp; {start_d} to {end_d}", styles["Normal"]))
    story.append(Spacer(1, 16))

    counts = {}
    for h in entries:
        counts[h["sentiment"]] = counts.get(h["sentiment"], 0) + 1
    summary_line = ", ".join(f"{k}: {v}" for k, v in counts.items())
    story.append(Paragraph("Mood summary", styles["Heading2"]))
    story.append(Paragraph(f"{len(entries)} entries logged. {summary_line}.", styles["Normal"]))
    story.append(Spacer(1, 12))

    story.append(Paragraph("Recommendation", styles["Heading2"]))
    story.append(Paragraph(recommendation_text, styles["Normal"]))
    story.append(Spacer(1, 16))

    story.append(Paragraph("Entries", styles["Heading2"]))
    table_data = [["Date", "Time", "Mood", "Emotion", "Confidence", "Source"]]
    for h in sorted(entries, key=lambda r: r["created_at"], reverse=True):
        table_data.append([
            str(h["mood_date"]),
            h["created_at"].strftime("%H:%M"),
            h["sentiment"] or "\u2014",
            h.get("emotion") or "\u2014",
            f"{h['confidence']:.0%}" if h.get("confidence") is not None else "\u2014",
            h["source"],
        ])
    tbl = Table(table_data, repeatRows=1, hAlign="LEFT")
    tbl.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1DBF73")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTSIZE", (0, 0), (-1, -1), 8),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#dddddd")),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f5f7f6")]),
    ]))
    story.append(tbl)

    doc.build(story)
    buf.seek(0)
    return buf.getvalue()


inject_css()

@st.cache_resource
def setup(): init_db()
setup()

if "page" not in st.session_state: st.session_state.page = "welcome"
if "show_auth_panel" not in st.session_state: st.session_state.show_auth_panel = False
if "auth_mode" not in st.session_state: st.session_state.auth_mode = "login"
if "token" not in st.session_state: st.session_state.token = None
if "email" not in st.session_state: st.session_state.email = None
if "chat_history" not in st.session_state: st.session_state.chat_history = []
if "cal_year" not in st.session_state: st.session_state.cal_year = date.today().year
if "cal_month" not in st.session_state: st.session_state.cal_month = date.today().month
if "today_mood_saved" not in st.session_state: st.session_state.today_mood_saved = False
if "nav" not in st.session_state: st.session_state.nav = "Home"

def goto_auth(mode): st.session_state.auth_mode = mode; st.rerun()

def valid_pw(pw):
    return len(pw) >= 8 and re.search(r"[A-Za-z]", pw) and re.search(r"[0-9]", pw)


if st.session_state.token:
    user = read_token(st.session_state.token)
    if user:
        role = user.get("role", "employee")
        headers = {"Authorization": f"Bearer {st.session_state.token}"}

        with st.sidebar:
            st.markdown(
                f"<div style='display:flex;align-items:center;gap:8px;padding:6px 4px 18px 4px'>"
                f"<span style='font-size:18px;font-weight:800;color:{INK}'>Mood Mentor</span></span>"
                f"</div>", unsafe_allow_html=True,
            )
            if role == "employee":
                nav_options = ["Home", "Journal", "Wellness Chat", "Face Detection", "Relax", "Dashboard"]
            else:
                nav_options = ["Reports"]
            st.session_state.nav = st.radio(
                "Navigate", nav_options,
                index=nav_options.index(st.session_state.nav) if st.session_state.nav in nav_options else 0,
                label_visibility="collapsed",
            )
            st.divider()
            st.caption(f"Signed in as **{user['username']}**")
            st.caption(f"{user['email']} · {role.capitalize()}")
            if st.button("Log out", use_container_width=True):
                st.session_state.token = None
                st.session_state.page = "welcome"
                st.session_state.show_auth_panel = False
                st.rerun()

        greeting = "Good Morning" if datetime.now().hour < 12 else (
            "Good Afternoon" if datetime.now().hour < 18 else "Good Evening")
        st.markdown(
            f"<div class='mm-header'><div><h2>{greeting}, {user['username']}!</h2>"
            f"<p>Here's your emotional wellness overview.</p></div></div>",
            unsafe_allow_html=True,
        )

        if role == "employee":
            section = st.session_state.nav

            if section == "Home":
                history_all = get_user_mood_history(user["id"], limit=500)
                latest = history_all[0] if history_all else None
                today_count = sum(1 for h in history_all if h["mood_date"] == date.today())
                streak = 0
                day_ptr = date.today()
                day_set = {h["mood_date"] for h in history_all}
                while day_ptr in day_set:
                    streak += 1
                    day_ptr = date.fromordinal(day_ptr.toordinal() - 1)

                positive_count = sum(1 for h in history_all if h["sentiment"] == "Happy")
                overall_score = int(100 * positive_count / len(history_all)) if history_all else 0

                m1, m2, m3, m4 = st.columns(4)
                with m1:
                    if latest:
                        s = style_for(latest["sentiment"])
                        metric_tile("Current Mood", f"{s['emoji']} {latest['sentiment']}")
                    else:
                        metric_tile("Current Mood", "—")
                with m2:
                    metric_tile("Overall Score", f"{overall_score}%", "Positive" if overall_score >= 50 else "Needs care")
                with m3:
                    metric_tile("Entries Today", today_count)
                with m4:
                    metric_tile("Current Streak", f"{streak} Days")

                st.write("")
                st.subheader("How Do You Feel?")
                now = datetime.now()
                st.caption(f"{now.strftime('%Y-%m-%d')}  {now.strftime('%H:%M')}")

                cols = st.columns(len(MOOD_LABELS))
                picked = st.session_state.get("picked_mood")
                for col, label in zip(cols, MOOD_LABELS):
                    s = style_for(label)
                    with col:
                        st.markdown(
                            f"<div style='text-align:center;font-size:36px'>{s['emoji']}</div>"
                            f"<div style='text-align:center;color:{s['color']};font-weight:600'>{label}</div>",
                            unsafe_allow_html=True,
                        )
                        if st.button("Select", key=f"pick_{label}", use_container_width=True):
                            st.session_state.picked_mood = label

                st.write("")
                confirm_col = st.columns([3, 1, 3])[1]
                with confirm_col:
                    disabled = picked is None
                    if st.button("Save mood", type="primary", disabled=disabled,
                                 use_container_width=True):
                        save_manual_mood(user["id"], st.session_state.picked_mood)
                        st.session_state.today_mood_saved = True
                        st.session_state.picked_mood = None
                        st.rerun()

                if st.session_state.today_mood_saved:
                    st.success("Today's mood saved!")
                    st.session_state.today_mood_saved = False
                st.markdown("</div>", unsafe_allow_html=True)

                st.subheader("Your Mood Calendar")

                nav_l, nav_mid, nav_r = st.columns([1, 3, 1])
                if nav_l.button("← Prev"):
                    m, y = st.session_state.cal_month - 1, st.session_state.cal_year
                    if m == 0: m, y = 12, y - 1
                    st.session_state.cal_month, st.session_state.cal_year = m, y
                    st.rerun()
                if nav_r.button("Next →"):
                    m, y = st.session_state.cal_month + 1, st.session_state.cal_year
                    if m == 13: m, y = 1, y + 1
                    st.session_state.cal_month, st.session_state.cal_year = m, y
                    st.rerun()
                nav_mid.markdown(
                    f"<h4 style='text-align:center'>{calendar.month_name[st.session_state.cal_month]} "
                    f"{st.session_state.cal_year}</h4>", unsafe_allow_html=True,
                )

                logs = get_mood_logs_for_month(user["id"], st.session_state.cal_year,
                                                st.session_state.cal_month)
                by_day = {row["mood_date"].day: row for row in logs}

                weeks = calendar.Calendar(firstweekday=6).monthdayscalendar(
                    st.session_state.cal_year, st.session_state.cal_month
                )
                day_names = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
                header_cols = st.columns(7)
                for c, name in zip(header_cols, day_names):
                    c.markdown(f"**{name}**")

                for week in weeks:
                    cols = st.columns(7)
                    for col, day_num in zip(cols, week):
                        if day_num == 0:
                            col.write("")
                            continue
                        entry = by_day.get(day_num)
                        s = style_for(entry["sentiment"] if entry else None)
                        time_label = entry["created_at"].strftime("%H:%M") if entry else ""
                        col.markdown(
                            f"<div title='{time_label}' style='text-align:center;padding:6px;border-radius:8px;"
                            f"background:{s['color']}22;border:1px solid {s['color']}'>"
                            f"<div style='font-size:11px'>{day_num}</div>"
                            f"<div style='font-size:20px'>{s['emoji']}</div>"
                            f"<div style='font-size:9px;color:#888'>{time_label}</div></div>",
                            unsafe_allow_html=True,
                        )

                legend = " · ".join(l for l in MOOD_LABELS)
                st.caption(f"{legend} · No entry logged  (hover/see time under each day)")
                st.markdown("</div>", unsafe_allow_html=True)

            elif section == "Journal":
                st.subheader(" Journal")
                journal_text = st.text_area(
                    "Write about how you're feeling today", height=150,
                    placeholder="Your note here...",
                )
                if st.button("Analyze my mood"):
                    if not journal_text.strip():
                        st.warning("Write something first.")
                    else:
                        with st.spinner("Running NLP analysis…"):
                            try:
                                resp = requests.post(
                                    f"{BACKEND_URL}/analyze-text",
                                    json={"text": journal_text},
                                    headers=headers, timeout=120,
                                )
                            except requests.exceptions.RequestException as e:
                                st.error(f"Could not reach backend: {e}"); resp = None
                        if resp is not None:
                            if resp.status_code != 200:
                                st.error("Analysis failed.")
                            else:
                                r = resp.json()
                                confidence = r.get("emotion_confidence")
                                save_mood_log(
                                    user["id"], r["final_sentiment"], r["final_emotion"],
                                    r["sentiment_scores"]["compound"], journal_text,
                                    confidence=confidence,
                                )
                                conf_str = f", Confidence: **{confidence:.0%}**" if confidence is not None else ""
                                st.success(f"Saved! Sentiment: **{r['final_sentiment']}**, "
                                           f"Emotion: **{r['final_emotion']}**{conf_str}")
                                st.bar_chart(r["emotion_scores"])
                                if r.get("recommendation"):
                                    st.info(f"**Recommendation:** {r['recommendation']}")
                st.markdown("</div>", unsafe_allow_html=True)

                st.subheader("Or upload a file")
                uploaded = st.file_uploader("Choose a CSV or TXT file", type=["csv", "txt"])
                if uploaded is not None and st.button("Run NLP Analysis on file"):
                    files = {"file": (uploaded.name, uploaded.getvalue())}
                    with st.spinner("Running multilingual NLP pipeline…"):
                        try:
                            resp = requests.post(f"{BACKEND_URL}/analyze", files=files,
                                                  headers=headers, timeout=120)
                        except requests.exceptions.RequestException as e:
                            st.error(f"Could not reach backend: {e}"); resp = None
                    if resp is not None:
                        if resp.status_code != 200:
                            st.error("Analysis failed.")
                        else:
                            r = resp.json()
                            confidence = r.get("emotion_confidence")
                            save_mood_log(
                                user["id"], r["final_sentiment"], r["final_emotion"],
                                r["sentiment_scores"]["compound"], r.get("cleaned_text", ""),
                                confidence=confidence,
                            )
                            conf_str = f", Confidence: **{confidence:.0%}**" if confidence is not None else ""
                            st.success(f"Saved! Sentiment: **{r['final_sentiment']}**, "
                                       f"Emotion: **{r['final_emotion']}**{conf_str}")
                            st.bar_chart(r["emotion_scores"])
                            if r.get("recommendation"):
                                st.info(f"**Recommendation:** {r['recommendation']}")
                st.markdown("</div>", unsafe_allow_html=True)

                st.subheader(" Past entries")
                history = [h for h in get_user_mood_history(user["id"], limit=20)
                           if h["journal_text"]]
                if not history:
                    st.caption("No journal entries yet.")
                for h in history:
                    s = style_for(h["sentiment"])
                    conf_str = f" · Confidence: {h['confidence']:.0%}" if h.get("confidence") is not None else ""
                    with st.expander(
                        f"{s['emoji']} {h['sentiment']} — {h['created_at'].strftime('%Y-%m-%d %H:%M')}{conf_str}"
                    ):
                        st.write(h["journal_text"])
                st.markdown("</div>", unsafe_allow_html=True)

            elif section == "Wellness Chat":
                st.subheader(" Wellness Chat")
                st.caption("A supportive space to talk about how you're feeling. "
                           "Not a substitute for professional care.")
                chat_box = st.container(height=450)
                with chat_box:
                    for turn in st.session_state.chat_history:
                        with st.chat_message(turn["role"]):
                            st.write(turn["content"])

                user_msg = st.chat_input("How are you feeling today?")
                if user_msg:
                    st.session_state.chat_history.append({"role": "user", "content": user_msg})
                    recent_history = st.session_state.chat_history[-10:-1]
                    try:
                        resp = requests.post(
                            f"{BACKEND_URL}/chat",
                            json={"message": user_msg, "history": recent_history},
                            headers=headers, timeout=60,
                        )
                        reply = resp.json()["reply"] if resp.status_code == 200 else \
                            "Sorry, I couldn't reach the wellness assistant right now."
                    except requests.exceptions.RequestException:
                        reply = "Sorry, I couldn't reach the wellness assistant right now."
                    st.session_state.chat_history.append({"role": "assistant", "content": reply})
                    st.rerun()

                if st.session_state.chat_history and st.button("Clear chat"):
                    st.session_state.chat_history = []
                    st.rerun()
                st.markdown("</div>", unsafe_allow_html=True)


            elif section == "Face Detection":
                st.markdown("<h1>📸 Face Scan & Recommendations</h1>", unsafe_allow_html=True)
                st.markdown("<p style='font-size:1.1rem;'>Using DeepFace AI to read your micro-expressions and provide personalized mentorship.</p>", unsafe_allow_html=True)

                # SAFE LAZY IMPORTS: Only load heavy AI if user clicks this tab!
                import cv2
                import numpy as np
                import os
                from deepface import DeepFace

                def analyze_and_display(image_bytes):
                    try:
                        nparr = np.frombuffer(image_bytes, np.uint8)
                        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
                        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                        st.info("Scanning biometric markers with MTCNN...")

                        tmp_path = "temp_scan.jpg"
                        cv2.imwrite(tmp_path, img)

                        results = DeepFace.analyze(img_path=tmp_path, actions=['emotion'], enforce_detection=True, detector_backend='mtcnn')
                        if not isinstance(results, list): results = [results]

                        st.success(f"Biometric Scan Complete! {len(results)} face(s) mapped.")

                        for i, face_data in enumerate(results):
                            emotion = face_data['dominant_emotion']
                            score_val = face_data['emotion'][emotion]
                            box = face_data['region']
                            x, y, w, h = box['x'], box['y'], box['w'], box['h']

                            # Draw Sci-Fi UI Elements
                            cv2.rectangle(img_rgb, (x, y), (x+w, y+h), (0, 255, 120), 3)
                            cv2.circle(img_rgb, (x, y), 5, (255, 255, 255), -1)
                            cv2.circle(img_rgb, (x+w, y), 5, (255, 255, 255), -1)
                            cv2.circle(img_rgb, (x, y+h), 5, (255, 255, 255), -1)
                            cv2.circle(img_rgb, (x+w, y+h), 5, (255, 255, 255), -1)
                            cv2.putText(img_rgb, f"{emotion.upper()} {score_val:.1f}%", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 120), 2)

                            save_face_scan(user["id"], emotion, float(score_val) / 100.0)

                        # Display Image
                        st.image(img_rgb, channels="RGB", use_container_width=True)

                        # MENTOR RECOMMENDATIONS
                        st.markdown("---")
                        st.markdown("<h3>🧠 Mood Mentor Analysis</h3>", unsafe_allow_html=True)
                        emotion = results[0]['dominant_emotion'].lower()
                        if emotion in ["happy", "joy", "amazing"]:
                            st.info("💡 **Mentor's Advice:** You're radiating positive energy! Channel this into your most challenging tasks today, or share your good mood by helping a colleague.")
                            st.balloons()
                        elif emotion in ["sad", "sadness"]:
                            st.warning("💡 **Mentor's Advice:** It's okay to feel down. Please take a moment for yourself. Try the 'Relax' tab for some guided breathing, or write your thoughts down in the Journal.")
                        elif emotion in ["angry", "anger", "disgust"]:
                            st.error("💡 **Mentor's Advice:** You seem frustrated. Step away from your screen for 5 minutes, get a glass of water, and try the 4-7-8 breathing technique in the Relax tab.")
                        elif emotion in ["fear", "surprise"]:
                            st.warning("💡 **Mentor's Advice:** Take a deep breath. Focus on what you can control right now. If you're feeling overwhelmed, break your tasks into smaller steps.")
                        else:
                            st.success("💡 **Mentor's Advice:** You seem balanced and focused. It's a great time to tackle deep work and maintain this calm state.")

                    except ValueError:
                        st.error("No face detected. Please ensure your face is clearly visible and try again.")
                    except Exception as e:
                        st.error(f"Error during scan: {e}")
                    finally:
                        if os.path.exists("temp_scan.jpg"): os.remove("temp_scan.jpg")

                tab1, tab2 = st.tabs(["📸 Camera Scanner", "📂 Upload Photo"])

                with tab1:
                    camera_photo = st.camera_input("Initiate Biometric Camera Scan")
                    if camera_photo: analyze_and_display(camera_photo.read())
                with tab2:
                    uploaded_image = st.file_uploader("Upload Image for Scanning", type=["jpg", "jpeg", "png"])
                    if uploaded_image: analyze_and_display(uploaded_image.read())

            elif section == "Relax":
                st.markdown("<h1>🧘 Smart Music Therapy & Breathing</h1>", unsafe_allow_html=True)
                st.markdown("<p style='font-size:1.1rem;'>Take a moment for yourself. Your mentor dynamically adjusts this page based on your recent mood entries.</p>", unsafe_allow_html=True)

                col1, col2 = st.columns([1,1], gap="large")

                with col1:
                    st.markdown("<h3>🌬️ Guided Breathing</h3>", unsafe_allow_html=True)
                    st.markdown("<p>Follow the circle. Breathe in as it expands, hold, and breathe out as it shrinks.</p>", unsafe_allow_html=True)
                    st.markdown('''
                        <div class="breathing-container">
                            <div class="circle">Breathe</div>
                        </div>
                    ''', unsafe_allow_html=True)
                    st.info("💡 **Mentor's Tip:** This 4-7-8 breathing technique activates your parasympathetic nervous system, reducing anxiety in just 60 seconds.")

                with col2:
                    st.markdown("<h3>🎵 Therapy Recommendations</h3>", unsafe_allow_html=True)

                    history = get_user_mood_history(user["id"], limit=1)
                    if history:
                        last_mood = history[0]["sentiment"]
                    else:
                        last_mood = "Normal"

                    # MENTOR RECOMMENDATIONS FOR RELAX PAGE
                    if last_mood in ["Sad", "Angry", "Stress", "Fear"]:
                        st.warning(f"**Mentor's Advice:** Since your last entry showed you were feeling **{last_mood}**, I recommend 5 minutes of mindful breathing before your next meeting. Listen to this calming acoustic playlist to help you center yourself.")
                        spotify_url = "https://open.spotify.com/embed/playlist/37i9dQZF1DWZqd5JICZI0u?utm_source=generator"
                    elif last_mood in ["Happy", "Amazing"]:
                        st.success(f"**Mentor's Advice:** Since your last entry showed you were feeling **{last_mood}**, keep that incredible momentum going! This upbeat playlist is perfect while you work.")
                        spotify_url = "https://open.spotify.com/embed/playlist/37i9dQZF1DXcBWIGoYBM5M?utm_source=generator"
                    else:
                        st.info(f"**Mentor's Advice:** You've been feeling **{last_mood}**. To help you find your flow and stay centered today, try this Lo-Fi Beats playlist.")
                        spotify_url = "https://open.spotify.com/embed/playlist/37i9dQZF1DWWQRwui0ExPn?utm_source=generator"

                    import streamlit.components.v1 as components
                    components.iframe(spotify_url, width=300, height=352, scrolling=False)

            elif section == "Dashboard":
                history = get_user_mood_history(user["id"], limit=200)
                if not history:
                    st.info("No entries yet — pick a mood on Home or write a journal entry to see your dashboard.")
                else:
                    counts = {label: 0 for label in MOOD_LABELS}
                    for h in history:
                        if h["sentiment"] in counts:
                            counts[h["sentiment"]] += 1

                    c1, c2 = st.columns(2)
                    with c1:
                        st.write("**Mood distribution**")
                        fig = donut_chart(counts)
                        if fig: st.pyplot(fig, use_container_width=False)
                        else: st.bar_chart(counts)
                        st.markdown("</div>", unsafe_allow_html=True)
                    with c2:
                        st.write("**Mood trend over time**")
                        by_date = {}
                        for h in history:
                            d = h["mood_date"]
                            by_date.setdefault(d, []).append(MOOD_TO_NUM.get(h["sentiment"], 0))
                        trend = {str(d): sum(v) / len(v) for d, v in sorted(by_date.items())}
                        st.line_chart(trend)
                        st.markdown("</div>", unsafe_allow_html=True)

                    st.write("**Emotions detected from journal entries**")
                    emo_counts = {}
                    for h in history:
                        if h["source"] == "nlp" and h["emotion"]:
                            emo_counts[h["emotion"]] = emo_counts.get(h["emotion"], 0) + 1
                    if emo_counts:
                        st.bar_chart(emo_counts)
                    else:
                        st.caption("No journal-based emotion data yet.")
                    st.markdown("</div>", unsafe_allow_html=True)

                    st.write("**Recent activity**")
                    table_rows = [{
                        "Date": h["mood_date"], "Time": h["created_at"].strftime("%H:%M"),
                        "Mood": f"{style_for(h['sentiment'])['emoji']} {h['sentiment']}",
                        "Confidence": f"{h['confidence']:.0%}" if h.get("confidence") is not None else "—",
                        "Source": h["source"],
                    } for h in history[:15]]
                    st.dataframe(table_rows, use_container_width=True)
                    st.markdown("</div>", unsafe_allow_html=True)
                    st.write("**Export report**")
                    oldest_date = history[-1]["mood_date"]
                    today = date.today()
                    date_range = st.date_input(
                        "Select date range", value=(oldest_date, today),
                        min_value=oldest_date, max_value=today,
                        key="dashboard_export_range",
                    )
                    if st.button("Export PDF"):
                        if isinstance(date_range, tuple) and len(date_range) == 2:
                            start_d, end_d = date_range
                        else:
                            start_d = end_d = date_range
                        filtered = [h for h in history if start_d <= h["mood_date"] <= end_d]
                        if not filtered:
                            st.warning("No entries in that date range.")
                        else:
                            recommendation_text = get_period_recommendation(filtered)
                            pdf_bytes = build_pdf_report(
                                user["username"], start_d, end_d, filtered, recommendation_text,
                            )
                            st.success(recommendation_text)
                            st.download_button(
                                "Download PDF", data=pdf_bytes,
                                file_name=f"moodmentor_report_{start_d}_{end_d}.pdf",
                                mime="application/pdf",
                            )
                    st.markdown("</div>", unsafe_allow_html=True)

        else:
            st.subheader("Employee Wellness Report")

            latest = get_latest_mood_per_employee()
            if not latest:
                st.info("No employee entries yet.")
            else:
                st.write("**Latest mood per employee**")
                table_rows = [{
                    "Employee": row["username"],
                    "Email": row["email"],
                    "Date": row["mood_date"],
                    "Time": row["created_at"].strftime("%H:%M"),
                    "Mood": f"{style_for(row['sentiment'])['emoji']} {row['sentiment']}",
                    "Emotion": row["emotion"],
                } for row in latest]
                st.dataframe(table_rows, use_container_width=True)
            st.markdown("</div>", unsafe_allow_html=True)

            st.write("**Team mood trend (last 30 days)**")
            history = get_all_employee_mood_logs(limit_days=30)
            if not history:
                st.info("Not enough data yet to draw a trend chart.")
            else:
                by_date = {}
                for row in history:
                    d = row["mood_date"]
                    by_date.setdefault(d, []).append(MOOD_TO_NUM.get(row["sentiment"], 0))
                trend = {str(d): sum(v) / len(v) for d, v in sorted(by_date.items())}
                st.line_chart(trend)
                st.caption("Average mood score per day across all employees "
                           "(2 = Happy, 0 = Neutral, -1 = Sad/Stress, -2 = Angry/Fear)")
            st.markdown("</div>", unsafe_allow_html=True)

        st.stop()
    st.session_state.token = None


if st.session_state.page == "welcome":

    if not st.session_state.show_auth_panel:
        st.markdown('<div class="welcome-box">', unsafe_allow_html=True)
        st.markdown("## Mood<span style='color:#eafff4'>Mentor</span>", unsafe_allow_html=True)
        st.markdown("#### AI-Powered Emotional Wellness Assistant")
        st.write(
            "Understand your emotions. Improve your well-being. Live your best life. "
            "Journey into your inner world through emojis, text, voice recordings, "
            "and notes — and watch your emotional landscape unfold through beautiful "
            "charts and insights."
        )
        st.markdown(
            "<div style='text-align:center;font-size:36px;padding:24px 0'>"
            "</div>",
            unsafe_allow_html=True,
        )
        st.markdown("</div>", unsafe_allow_html=True)
        st.write("")
        if st.button("Get Started →", type="primary", use_container_width=True):
            st.session_state.show_auth_panel = True
            st.rerun()
        st.stop()

    left, right = st.columns([3, 2])

    with left:
        st.markdown('<div class="welcome-box">', unsafe_allow_html=True)
        st.markdown("## Mood<span style='color:#eafff4'>Mentor</span>", unsafe_allow_html=True)
        st.markdown("#### AI-Powered Emotional Wellness Assistant")
        st.write(
            "Understand your emotions. Improve your well-being. Live your best life. "
            "Journey into your inner world through emojis, text, voice recordings, "
            "and notes — and watch your emotional landscape unfold through beautiful "
            "charts and insights."
        )
        st.markdown(
            "<div style='text-align:center;font-size:36px;padding:24px 0'>"
            "</div>",
            unsafe_allow_html=True,
        )
        st.markdown("</div>", unsafe_allow_html=True)

    with right:
        st.markdown('<div class="auth-card">', unsafe_allow_html=True)
        mode = st.session_state.auth_mode

        if mode == "login":
            st.markdown("### Welcome Back!")
            st.caption("Login to your account")
            with st.form("login"):
                email = st.text_input("Email", placeholder="Enter your email")
                pw = st.text_input("Password", type="password", placeholder="Enter your password")
                go = st.form_submit_button("Login", type="primary", use_container_width=True)
            if go:
                u = get_user(email.strip().lower())
                if not u or not check_pw(pw, u["password_hash"]):
                    st.error("Invalid email or password.")
                elif not u["is_verified"]:
                    st.warning("Verify your email first.")
                    st.session_state.email = u["email"]; goto_auth("verify")
                else:
                    st.session_state.token = make_token(u)
                    st.rerun()
            c1, c2 = st.columns(2)
            if c1.button("Sign up", use_container_width=True): goto_auth("signup")
            if c2.button("Forgot password?", use_container_width=True): goto_auth("forgot")

        elif mode == "signup":
            st.markdown("### Create Account")
            st.caption("Let's get you started")
            with st.form("signup"):
                username = st.text_input("Full Name", placeholder="Enter your full name")
                email = st.text_input("Email", placeholder="Enter your email")
                pw = st.text_input("Password", type="password", placeholder="Create password")
                role_label = st.radio("I am signing up as a:", ["Employee", "Manager"], horizontal=True)
                go = st.form_submit_button("Send OTP", type="primary", use_container_width=True)
            if go:
                email = email.strip().lower()
                role = "manager" if role_label == "Manager" else "employee"
                if len(username) < 3:
                    st.error("Username too short.")
                elif not valid_pw(pw):
                    st.error("Password needs 8+ chars, letters and numbers.")
                elif username_taken(username) or get_user(email):
                    st.error("Username or email already in use.")
                else:
                    create_user(username, email, pw, role=role)
                    code = new_otp(); save_otp(email, code, "signup")
                    ok, msg = send_otp(email, code, "signup")
                    if ok:
                        st.session_state.email = email
                        st.success("Check your email for the code.")
                        goto_auth("verify")
                    else:
                        st.error(f"Email failed: {msg}")
            if st.button("Already have an account? Login"): goto_auth("login")

        elif mode == "verify":
            email = st.session_state.email
            st.markdown("### Verify OTP")
            st.caption(f"We have sent a 6-digit code to {email}")
            with st.form("verify"):
                code = st.text_input("Code", max_chars=6, placeholder="Enter 6-digit code")
                go = st.form_submit_button("Verify OTP", type="primary", use_container_width=True)
            if go:
                if check_otp(email, code.strip(), "signup"):
                    verify_user(email)
                    st.success("Verified! Please log in.")
                    goto_auth("login")
                else:
                    st.error("Invalid or expired code.")
            if st.button("← Back to login"): goto_auth("login")

        elif mode == "forgot":
            st.markdown("### Forgot password")
            with st.form("forgot"):
                email = st.text_input("Your account email")
                go = st.form_submit_button("Send reset code", type="primary", use_container_width=True)
            if go:
                email = email.strip().lower()
                if get_user(email):
                    code = new_otp(); save_otp(email, code, "password_reset")
                    send_otp(email, code, "password_reset")
                st.session_state.email = email
                st.info("If that email exists, a code was sent.")
                goto_auth("reset")
            if st.button("← Back to login"): goto_auth("login")

        elif mode == "reset":
            email = st.session_state.email
            st.markdown("### Reset password")
            with st.form("reset"):
                code = st.text_input("Reset code", max_chars=6)
                pw = st.text_input("New password", type="password")
                go = st.form_submit_button("Reset", type="primary", use_container_width=True)
            if go:
                if not valid_pw(pw):
                    st.error("Password needs 8+ chars, letters and numbers.")
                elif not check_otp(email, code.strip(), "password_reset"):
                    st.error("Invalid or expired code.")
                else:
                    set_password(email, pw)
                    st.success("Password reset. Please log in.")
                    goto_auth("login")
            if st.button("← Back to login"): goto_auth("login")

        st.markdown("</div>", unsafe_allow_html=True)

    st.stop()



Writing app.py


In [9]:
%%writefile nlp_pipeline.py
"""
nlp_pipeline.py
Multilingual NLP pipeline for employee feedback:
normalize -> detect language -> clean -> tokenize -> stopword-filter ->
translate to English -> lemmatize -> sentiment (VADER) -> emotion (BERT).

Stopword filtering uses the `stopwordsiso` package, which ships stopword
sets for 50+ languages keyed by ISO 639-1 code (the same codes langdetect
returns), so any supported language is handled automatically instead of
needing a hardcoded list per language. If the detected language isn't in
stopwordsiso's coverage, filtering is simply skipped for that text.

Heavy libs (spacy model, translator, vader, BERT emotion model, Qwen chat
model) load once at import time via lazy module-level globals, so repeated
/analyze calls reuse them.
"""

import re
import ftfy
import emoji
import spacy
import torch
import stopwordsiso
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline as hf_pipeline,
)
from langdetect import detect, DetectorFactory
from deep_translator import GoogleTranslator
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from recommendations import get_recommendation, WELLNESS_RECOMMENDATIONS

DetectorFactory.seed = 0

_nlp = None
_vader = None
_qwen_model = None
_qwen_tokenizer = None
_bert_emotion_pipeline = None

QWEN_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

BERT_EMOTION_MODEL_NAME = "bhadresh-savani/bert-base-go-emotion"

LANGUAGE_NAMES = {
    "te": "Telugu", "kn": "Kannada", "en": "English", "ta": "Tamil",
    "hi": "Hindi", "ml": "Malayalam", "mr": "Marathi", "bn": "Bengali", "gu": "Gujarati",
    "fr": "French", "de": "German", "es": "Spanish", "pt": "Portuguese",
    "ar": "Arabic", "zh": "Chinese", "ja": "Japanese", "ko": "Korean", "ru": "Russian",
}


def _get_stopwords(language_code: str) -> set:
    """
    Returns the stopword set for `language_code` using stopwordsiso, which
    covers 50+ languages by ISO 639-1 code. Returns an empty set for any
    language it doesn't cover -- filtering is skipped rather than failing,
    so unsupported languages still flow through the rest of the pipeline.
    """
    if stopwordsiso.has_lang(language_code):
        return stopwordsiso.stopwords(language_code)
    return set()

EMOTION_LABELS = ["Happy", "Sad", "Stress", "Angry", "Fear", "Neutral"]


GOEMOTIONS_TO_APP_LABEL = {
    "joy": "Happy", "amusement": "Happy", "excitement": "Happy",
    "love": "Happy", "gratitude": "Happy", "optimism": "Happy",
    "relief": "Happy", "pride": "Happy", "admiration": "Happy",
    "approval": "Happy", "caring": "Happy",

    "sadness": "Sad", "disappointment": "Sad", "grief": "Sad",
    "remorse": "Sad",

    "nervousness": "Stress", "embarrassment": "Stress",
    "confusion": "Stress",

    "anger": "Angry", "annoyance": "Angry", "disgust": "Angry",
    "disapproval": "Angry",

    "fear": "Fear",

    "neutral": "Neutral", "realization": "Neutral", "surprise": "Neutral",
    "curiosity": "Neutral", "desire": "Neutral",
}




def _get_nlp():
    """Lazy-load the multilingual spaCy model once per process."""
    global _nlp
    if _nlp is None:
        _nlp = spacy.load("xx_sent_ud_sm")
    return _nlp


def _get_vader():
    global _vader
    if _vader is None:
        _vader = SentimentIntensityAnalyzer()
    return _vader


def _get_qwen():
    """Lazy-load Qwen2.5-0.5B-Instruct once per process (GPU if available).
    Still used by the wellness chatbot (wellness_chat_reply) -- only the
    emotion-detection step now uses BERT instead."""
    global _qwen_model, _qwen_tokenizer
    if _qwen_model is None:
        _qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
        _qwen_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL_NAME,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
        )
    return _qwen_model, _qwen_tokenizer


def _get_bert_emotion_pipeline():
    """
    Lazy-load the fine-tuned BERT emotion classifier once per process, using
    Hugging Face's `pipeline()` helper -- this bundles the tokenizer and the
    model together so we just call it with raw text and get scores back.

    `top_k=None` tells the pipeline to return a score for every label
    instead of just the single top prediction, so we can build a full
    scores dict (matching what the UI already expects).
    """
    global _bert_emotion_pipeline
    if _bert_emotion_pipeline is None:
        _bert_emotion_pipeline = hf_pipeline(
            "text-classification",
            model=BERT_EMOTION_MODEL_NAME,
            top_k=None,
            device=0 if torch.cuda.is_available() else -1,
        )
    return _bert_emotion_pipeline


def _bert_emotion(text: str) -> dict:
    """
    Classifies `text` using the fine-tuned BERT GoEmotions model, then maps
    the 28 GoEmotions labels down to our 6 app-level EMOTION_LABELS by
    summing mapped scores. Returns the same shape the rest of the app
    already expects: {"emotion": <label>, "scores": {label: 0-1, ...}}.
    """
    classifier = _get_bert_emotion_pipeline()

    if not text.strip():
        text = "(empty feedback)"

    raw_predictions = classifier(text, truncation=True)[0]

    app_scores = {label: 0.0 for label in EMOTION_LABELS}
    for pred in raw_predictions:
        goemotion_label = pred["label"].lower()
        app_label = GOEMOTIONS_TO_APP_LABEL.get(goemotion_label, "Neutral")
        app_scores[app_label] += pred["score"]

    total = sum(app_scores.values()) or 1.0
    app_scores = {label: round(score / total, 4) for label, score in app_scores.items()}

    final_emotion = max(app_scores, key=app_scores.get)
    confidence = app_scores[final_emotion]
    return {"emotion": final_emotion, "scores": app_scores, "confidence": confidence}


def process_employee_feedback(text: str) -> dict:
    """Runs the full pipeline on a single blob of text and returns a results dict."""
    nlp = _get_nlp()
    vader = _get_vader()

    normalized_text = ftfy.fix_text(text)

    try:
        language = detect(normalized_text)
    except Exception:
        language = "unknown"
    detected_language = LANGUAGE_NAMES.get(language, "Other / Unknown")

    emoji_list = [ch for ch in normalized_text if ch in emoji.EMOJI_DATA]

    cleaned_text = re.sub(r"https?://\S+|www\.\S+", " ", normalized_text)
    cleaned_text = re.sub(r"\S+@\S+", " ", cleaned_text)
    cleaned_text = re.sub(r"@\w+|#\w+", " ", cleaned_text)
    cleaned_text = emoji.replace_emoji(cleaned_text, replace="")
    cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

    doc = nlp(cleaned_text)
    sentences = [s.text.strip() for s in doc.sents if s.text.strip()]
    original_tokens = [t.text for t in doc if not t.is_space]
    clean_tokens = [t.text for t in doc if not t.is_punct and not t.is_space and not t.like_num]

    selected_stopwords = _get_stopwords(language)
    filtered_tokens = [t for t in clean_tokens if t.lower() not in selected_stopwords]
    final_preprocessed_text = " ".join(filtered_tokens)

    try:
        translated_text = GoogleTranslator(source="auto", target="en").translate(final_preprocessed_text)
    except Exception as error:
        translated_text = f"Translation failed: {error}"

    english_doc = nlp(translated_text)
    lemmas = [t.lemma_ if t.lemma_ else t.text for t in english_doc if not t.is_space]
    lemmatized_text = " ".join(lemmas)

    sentiment_scores = vader.polarity_scores(translated_text)
    compound_score = sentiment_scores["compound"]
    if compound_score >= 0.05:
        final_sentiment = "Positive"
    elif compound_score <= -0.05:
        final_sentiment = "Negative"
    else:
        final_sentiment = "Neutral"

    bert_result = _bert_emotion(translated_text)
    emotion_scores = bert_result["scores"]
    final_emotion_label = bert_result["emotion"]
    emotion_confidence = bert_result["confidence"]

    # BERT's "Neutral" bucket sums 5 GoEmotions sub-labels (neutral,
    # realization, surprise, curiosity, desire) vs. 1-4 for the other
    # buckets, so on short/ambiguous text it wins the argmax by default
    # even when VADER clearly reads the text as negative (e.g. "stressed").
    # Previously this mismatch was only patched inside get_recommendation()
    # for the *recommendation text* -- the emotion label shown to the user
    # and saved to mood_logs.emotion stayed "Neutral", contradicting the
    # sentiment shown right next to it and skewing the Dashboard's
    # "Emotions detected" chart toward Neutral. Apply the same override to
    # final_emotion itself so what's displayed/stored/charted is consistent
    # with what's recommended.
    if final_emotion_label == "Neutral" and final_sentiment == "Negative":
        final_emotion_label = "Sad"

    final_emotion = final_emotion_label
    recommendation = get_recommendation(final_emotion_label, emotion_confidence, final_sentiment, compound_score)

    return {
        "language_code": language,
        "detected_language": detected_language,
        "normalized_text": normalized_text,
        "cleaned_text": cleaned_text,
        "sentences": sentences,
        "original_tokens": original_tokens,
        "filtered_tokens": filtered_tokens,
        "emoji_list": emoji_list,
        "final_preprocessed_text": final_preprocessed_text,
        "translated_text": translated_text,
        "lemmatized_text": lemmatized_text,
        "sentiment_scores": sentiment_scores,
        "final_sentiment": final_sentiment,
        "emotion_scores": emotion_scores,
        "final_emotion": final_emotion,
        "emotion_confidence": emotion_confidence,
        "recommendation": recommendation,
    }


CRISIS_KEYWORDS = [
    "suicide", "kill myself", "end my life", "want to die", "self harm",
    "self-harm", "hurt myself", "not worth living", "no reason to live",
]

CRISIS_MESSAGE = (
    "I'm really glad you reached out, and I want to make sure you get support "
    "beyond what I can offer here. If you're in immediate danger, please contact "
    "your local emergency number right now. You can also reach a crisis line: "
    "in India, AASRA is available at +91-9820466726 (24/7). If you're outside "
    "India, please look up a local crisis helpline or talk to a trusted person "
    "or your HR/EAP contact. You don't have to go through this alone."
)

WELLNESS_SYSTEM_PROMPT = (
    "You are a supportive workplace wellness assistant for employees. "
    "Your role is to listen, validate feelings, and offer general, gentle "
    "coping suggestions (like breathing exercises, taking a short break, "
    "or talking to a trusted colleague or manager). "
    "You are NOT a therapist or doctor: never diagnose any condition, never "
    "claim expertise you don't have, and never give medical or medication "
    "advice. If the employee describes something serious (ongoing crisis, "
    "self-harm, harming others), gently encourage them to contact a mental "
    "health professional, their HR/EAP program, or a crisis helpline. "
    "Keep replies short (2-4 sentences), warm, and non-judgmental. "
    "Avoid clinical labels and avoid being preachy or repetitive."
)


def _contains_crisis_language(text: str) -> bool:
    lowered = text.lower()
    return any(kw in lowered for kw in CRISIS_KEYWORDS)


def wellness_chat_reply(message: str, history: list[dict] | None = None) -> dict:
    """
    Generates a supportive wellness chatbot reply using the Qwen chat model.
    (The chatbot still uses Qwen -- it needs to generate free-form
    conversational replies, which is a generation task, not a
    classification task, so BERT isn't a fit here.)

    `history` is an optional list of {"role": "user"|"assistant", "content": str}
    dicts representing prior turns in the conversation (kept short/recent by
    the caller — this function does not trim it).

    Always checks for crisis language first; if found, returns a fixed,
    resource-pointing message instead of an LLM-generated one, since we
    never want a small model improvising in a safety-critical moment.
    """
    if _contains_crisis_language(message):
        return {"reply": CRISIS_MESSAGE, "flagged": True}

    model, tokenizer = _get_qwen()

    messages = [{"role": "system", "content": WELLNESS_SYSTEM_PROMPT}]
    for turn in (history or []):
        if turn.get("role") in ("user", "assistant") and turn.get("content"):
            messages.append({"role": turn["role"], "content": turn["content"]})
    messages.append({"role": "user", "content": message})

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    reply = tokenizer.decode(generated, skip_special_tokens=True).strip()

    if not reply:
        reply = "I'm here and listening — could you tell me a bit more about how you're feeling?"

    return {"reply": reply, "flagged": False}



Writing nlp_pipeline.py


In [10]:
from db import cursor

with cursor(commit=True) as cur:
    # 1. Normalize any case-variant spellings of Neutral to the canonical form.
    cur.execute(
        "UPDATE mood_logs SET emotion = 'Neutral' "
        "WHERE emotion ILIKE 'neutral' AND emotion != 'Neutral'"
    )
    normalized = cur.rowcount

    # 2. Re-apply the Neutral+Negative -> Sad override to rows saved before the fix.
    #    sentiment here is the mapped 5-point label ('Sad'), set by NLP_TO_MOOD_LABEL
    #    from the pipeline's original 'Negative' sentiment -- see db.save_mood_log().
    cur.execute(
        "UPDATE mood_logs SET emotion = 'Sad' "
        "WHERE emotion = 'Neutral' AND sentiment = 'Sad' AND source = 'nlp'"
    )
    relabeled = cur.rowcount

print(f"Normalized casing on {normalized} row(s); relabeled {relabeled} Neutral->Sad row(s).")

Normalized casing on 0 row(s); relabeled 0 Neutral->Sad row(s).


In [11]:
%%writefile backend.py
import os, io, jwt, csv
from fastapi import FastAPI, UploadFile, File, Form, Header, HTTPException
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv
from nlp_pipeline import process_employee_feedback, wellness_chat_reply
load_dotenv()

SECRET = os.getenv("JWT_SECRET")
app = FastAPI(title="Upload API")

app.add_middleware(CORSMiddleware, allow_origins=["*"],
                    allow_methods=["*"], allow_headers=["*"])

def get_user(authorization: str = Header(None)):
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(401, "Missing token")
    token = authorization.split(" ", 1)[1]
    try:
        return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError:
        raise HTTPException(401, "Invalid or expired token")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/upload")
async def upload(file: UploadFile = File(...), authorization: str = Header(None)):
    user = get_user(authorization)

    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext not in ("csv", "txt"):
        raise HTTPException(400, "Only .csv or .txt files are allowed.")

    raw = await file.read()
    max_bytes = 5 * 1024 * 1024
    if len(raw) > max_bytes:
        raise HTTPException(400, "File too large (max 5 MB).")

    try:
        text = raw.decode("utf-8")
    except UnicodeDecodeError:
        raise HTTPException(400, "File must be UTF-8 text.")

    lines = text.splitlines()
    row_count = len(lines)
    preview_lines = lines[:20]

    columns = None
    preview_rows = None
    if ext == "csv":
        reader = csv.reader(io.StringIO(text))
        rows = list(reader)
        if rows:
            columns = rows[0]
            preview_rows = rows[1:21]
            row_count = max(len(rows) - 1, 0)

    return {
        "filename": name,
        "type": ext,
        "uploaded_by": user["username"],
        "row_count": row_count,
        "columns": columns,
        "preview_rows": preview_rows,
        "preview_lines": None if ext == "csv" else preview_lines,
    }


def _extract_text_blob(raw: bytes, ext: str, column: str | None) -> tuple[str, str | None]:
    """
    Returns (text_blob, used_column). For TXT, used_column is None.
    For CSV, joins all non-empty values of the chosen column (or the last
    column if none/invalid was specified) into one whitespace-joined blob —
    matches the notebook's "whole file as one blob" behavior.
    """
    text = raw.decode("utf-8")

    if ext == "txt":
        return text.strip(), None

    reader = csv.reader(io.StringIO(text))
    rows = list(reader)
    if not rows:
        raise HTTPException(400, "CSV file has no rows.")

    header = rows[0]
    data_rows = rows[1:]
    if not data_rows:
        raise HTTPException(400, "CSV file has a header but no data rows.")

    col_index = None
    if column and column in header:
        col_index = header.index(column)
    else:
        col_index = len(header) - 1

    values = [row[col_index] for row in data_rows if len(row) > col_index and row[col_index].strip()]
    blob = " ".join(values).strip()
    if not blob:
        raise HTTPException(400, f"Column '{header[col_index]}' has no readable text.")
    return blob, header[col_index]


@app.post("/analyze")
async def analyze(file: UploadFile = File(...), column: str = Form(None),
                   authorization: str = Header(None)):
    """
    Runs the multilingual NLP pipeline (language detection, cleaning,
    stopword filtering, translation, lemmatization, VADER sentiment,
    keyword-based emotion) on an uploaded .csv or .txt file.
    """
    get_user(authorization)

    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext not in ("csv", "txt"):
        raise HTTPException(400, "Only .csv or .txt files are allowed.")

    raw = await file.read()
    max_bytes = 5 * 1024 * 1024
    if len(raw) > max_bytes:
        raise HTTPException(400, "File too large (max 5 MB).")

    try:
        text_blob, used_column = _extract_text_blob(raw, ext, column)
    except UnicodeDecodeError:
        raise HTTPException(400, "File must be UTF-8 text.")

    results = process_employee_feedback(text_blob)
    results["filename"] = name
    results["file_type"] = ext.upper()
    results["used_column"] = used_column
    results["original_char_count"] = len(text_blob)
    return results



class TextIn(BaseModel):
    text: str

@app.post("/analyze-text")
async def analyze_text(payload: TextIn, authorization: str = Header(None)):
    """Same NLP pipeline as /analyze, but for text typed directly into the
    Journal tab's textbox instead of an uploaded file."""
    get_user(authorization)

    text_blob = payload.text.strip()
    if not text_blob:
        raise HTTPException(400, "Text cannot be empty.")

    results = process_employee_feedback(text_blob)
    results["filename"] = None
    results["file_type"] = "TEXT"
    results["used_column"] = None
    results["original_char_count"] = len(text_blob)
    return results

class ChatTurn(BaseModel):
    role: str
    content: str


class ChatRequest(BaseModel):
    message: str
    history: list[ChatTurn] = []


@app.post("/chat")
async def chat(payload: ChatRequest, authorization: str = Header(None)):
    """
    Wellness support chatbot endpoint. Stateless on the server: the client
    (Streamlit) sends the recent conversation history along with each new
    message, and we generate the next reply with the same Qwen model used
    for emotion detection.
    """
    get_user(authorization)

    message = payload.message.strip()
    if not message:
        raise HTTPException(400, "Message cannot be empty.")

    history = [turn.dict() for turn in payload.history]
    result = wellness_chat_reply(message, history=history)
    return result


Writing backend.py


In [12]:
from db import init_db
init_db()
print("Connected to PostgreSQL and ensured tables exist.")

Connected to PostgreSQL and ensured tables exist.


In [15]:
from pyngrok import ngrok, conf
import subprocess, time

conf.get_default().auth_token = values["NGROK_AUTHTOKEN"]

# Kill any previous tunnels/streamlit/uvicorn instances from earlier runs in this session
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
get_ipython().system_raw('pkill -f uvicorn || true')
time.sleep(1)

# Launch FastAPI (backend.py) in the background on port 8000 (internal only, not tunneled)
get_ipython().system_raw(
    'uvicorn backend:app --host 0.0.0.0 --port 8000 &'
)
time.sleep(5)
# Launch Streamlit in the background, quietly, on port 8501
get_ipython().system_raw(
    'streamlit run app.py --server.port 8501 --server.headless true '
    '--server.enableCORS false --server.enableXsrfProtection false &'
)
time.sleep(4)  # give both servers a moment to boot

public_url = ngrok.connect(8501, "http")
print(f" Your app is live at: {public_url}")

 Your app is live at: NgrokTunnel: "https://pamperer-twilight-paper.ngrok-free.dev" -> "http://localhost:8501"


In [14]:
from pyngrok import ngrok
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
get_ipython().system_raw('pkill -f uvicorn || true')
print("Stopped Streamlit, FastAPI, and closed ngrok tunnel.")

Stopped Streamlit, FastAPI, and closed ngrok tunnel.
